In [1]:
import torch
import torch.nn as nn
from pathlib import Path
import warnings
warnings.simplefilter("ignore", UserWarning)
import mltrainer
mltrainer.__version__

'0.2.4'

Lets get some data

In [ ]:
""" 
source:                 https://pypi.org/project/mads-datasets/ 
create_factory()        a dataset facroty dynamically generates, structures, or manages data collections (in this case the FASHION dataset from MNIST). With the factory, 
                        you can download the data, create datasets and provide the datasets wrapped in datastreamers in one command
create_datastreamer()   creates a data streamer: tool to process continuous flow of data (in batches)
"""

from mads_datasets import DatasetFactoryProvider, DatasetType
from mltrainer.preprocessors import BasePreprocessor
preprocessor = BasePreprocessor()

fashionfactory = DatasetFactoryProvider.create_factory(DatasetType.FASHION)
streamers = fashionfactory.create_datastreamer(batchsize=64, preprocessor=preprocessor)
# flowersfactory = DatasetFactoryProvider.create_factory(DatasetType.FLOWERS)
# streamers = flowersfactory.create_datastreamer(batchsize=32, preprocessor=preprocessor)
train = streamers["train"]
valid = streamers["valid"]

2026-05-31 13:25:31.183 | INFO     | mads_datasets.base:download_data:121 - Folder already exists at /home/gjongde/.cache/mads_datasets/fashionmnist
2026-05-31 13:25:31.186 | INFO     | mads_datasets.base:download_data:124 - File already exists at /home/gjongde/.cache/mads_datasets/fashionmnist/fashionmnist.pt


In [ ]:
""" 
length of both the training and the validation set
"""

len(train), len(valid)

(937, 156)

We can obtain an item:

In [ ]:
"""   
.stream()               will return a generator that yields batches of data
result explained:       (torch.Size([64, 1, 28, 28]), torch.Size([64]))
                        
                        torch.Size[64, 1, 28, 28] = x
                        64 = batchsize (in this case the number of images)
                        1  = channel (1 = greyscale, 3 would be a colored image (RGB))
                        28 = width
                        28 = height

                        torch.Size([64]) = y
                        64 = number of elements 
"""

trainstreamer = train.stream()
validstreamer = valid.stream()
x, y = next(iter(trainstreamer))
x.shape, y.shape

(torch.Size([64, 1, 28, 28]), torch.Size([64]))

The image follows the channels-first convention: (channel, width, height). The label is an integer.

Lets pull this through a Conv2d layer:

In [ ]:
"""  
Conv2d layer:           A 2D Convolutional layer used in CNNs used primarily for computer vision
"""

in_channels = x.shape[1]

In [ ]:
"""  
CNNs                    designed to process image data by capturing spatial relationships between pixels

torch.nn.Conv2d         class which applies 2D convolution over an input signal 
in_channels             number of channels in the input image (each channel represents a color)
out_channels            number of channels produced by the convolution
kernel_size             size of the convolving kernel. In CNN, kernels (small matrices used for feature extraction) are filters which helps 
                        CNNs to extract important features from images such as edges, textures and patterns
padding                 padding added to all four sides of the input. Default = 0. Padding refers to adding extra rows and columns of pixels
                        around the border of the image before passing it through a convolutional filter
"""

conv = nn.Conv2d(
    in_channels=in_channels,
    out_channels=64,    # ahaa, hierdoor heeft de output 64 channels
    kernel_size=3,
    padding=(1,1))
out = conv(x)
out.shape

torch.Size([64, 64, 28, 28])

What is happening here? Can you explain all the parameters, and relate them to the outputshape?

Let's see what happens if we change the padding:

In [ ]:
"""  
When the padding is adapted to (0, 0) instead of (1, 1) the width and height of the output decrease
Hypothesis: I think this happens because without padding, there's loss of border information, since convolution reduces output size
"""

conv = nn.Conv2d(
    in_channels=in_channels,
    out_channels=64,
    kernel_size=3,
    padding=(0,0))
out = conv(x)
out.shape

torch.Size([64, 64, 26, 26])

And if we change the stride from the default 1 to 2:

In [ ]:
"""   
stride                  stride of the convolution. A stride is the number of pixels a kernel (filter) moves as it slides across an 
                        input image. It controls how the filter scans the data, fundamentally impacting the size of the output and 
                        the features the network captures

When the stride is adapted to 2 instead of 1, the resulting width and height are halved.
"""

conv = nn.Conv2d(
    in_channels=in_channels,
    out_channels=64,
    kernel_size=3,
    padding=(1,1),
    stride=2)
out = conv(x)
out.shape

torch.Size([64, 64, 14, 14])

As you can see, you need to think about what is going in and out of the convolution. We can stitch multiple layers together like this:

In [ ]:
"""  
INPUT_UNTILL____________________________________________________|_RESULTING_SHAPE___________|_WHAT_IS_HAPPENING____|
x.shape                                                         | [64,  1, 28, 28]          | original
nn.Conv2d(in_channels, 32, kernel_size=3, stride=1, padding=1)  | [64, 32, 28, 28]          | output channels = 32 due to parameter out_channels = 32
nn.ReLU()                                                       | [64, 32, 28, 28]          | values below zero become 0, no effect on size
nn.MaxPool2d(kernel_size=2)                                     | [64, 32, 14, 14]          | a pooling layer is used to reduce dimensions while keeping 
.                                                               .                           . the most important information. MaxPooling selects the 
.                                                               .                           . maximum value from each region
nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=0)           | [64, 32, 12, 12]          | reduction due to padding?
nn.ReLU()                                                       | [64, 32, 12, 12]          | values below zero become 0, no effect on size
nn.MaxPool2d(kernel_size=2)                                     | [64, 32,  6,  6]          | maximum values for each region are selected
nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=0)           | [64, 32,  4,  4]          | again, padding?
nn.ReLU()                                                       | [64, 32,  4,  4]          | values below zero become 0, no effect on size
nn.MaxPool2d(kernel_size=2)                                     | [64, 32,  2,  2]          | maximum values for each region are selected
"""

convolutions = nn.Sequential(
    nn.Conv2d(in_channels, 32, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=0),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=0),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2)
)
out = convolutions(x)
out.shape

torch.Size([64, 32, 2, 2])

As you can see, the dimensions of the featuremap have become really small. You need to take this into account: If we would have started with a smaller image, we could get errors...

In [ ]:
"""   
Klinkt logisch
"""

x_too_small = torch.rand((32, 1, 12, 12))

try:
    convolutions(x_too_small)
except RuntimeError as err:
    print("ERROR:", err)

ERROR: Calculated padded input size per channel: (2 x 2). Kernel size: (3 x 3). Kernel size can't be greater than actual input size


At this point our `out` has 32 activation maps, each 2x2 big.

If we want to pull the activation maps through a neural network (A dense layer) we will need to flatten them (do you understand what happens if you dont do that?)

In [ ]:
"""  
NOTE TO SELF CONCERNING CHANNELS / ACTIVATION MAPS

Alrighty, so at first x = torchSize([64, 1, 28, 28]), where 1 equals a single greyscale channel

And in output out = torchSize([64, 32,  2,  2]), 32 is referred to as 'activation maps'

According to some research these terms do not mean the same, so what happened?

    Channels define the depth of the data. In an input layer this can be RGB or greyscale. In a hidden layer this can mean the number of filters the network is using.
    Activation maps is the 2d spatial output produced by applying a single filter across previously mentioned channels.

spatial output:        "A spatial output refers to predictions or representations that preserve the physical, 
                        locational, or structural geometry of the input. Rather than compressing data into a 
                        single label or value, the output retains a multi-dimensional grid format (like an 
                        image or a matrix) that maps directly to locations"


"""

In [24]:
input_nn = nn.Flatten()(out)
input_nn.shape

torch.Size([64, 128])

Note that there are potential problems connecting the image layers and the linear layers:
- Conv2d and MaxPool both expect 4 dimensional data (batch, channels/activationmaps, width, height)
- Linear layers expect 2 dimensional data (batch, features)
- Linear layers wont crash if you feed them data with more dimensions! However, they will just work on the last dimension, and thats probably not what you want.

This means we need to somehow transform the 4D data into 2D. There are some options here:
- Some sort of aggregation; the activationmaps are typically small (eg 2x2) and they indicate that the filter has detected a features. There are a lot of different ways to aggregate this: mean, max, min, sum, etc...
- Flatten: a flatten layer simple transforms (batch, C, W, H) into (batch, C * W * H). lets say you have (32, 32, 2, 2) than after a flatten you end up with (32, 128). The problem here is, when you use a different amount of Conv2d layers, or a different stride or padding, you will end up with a different size of activationmap, eg (32, 32, 3, 3), which would mean you would end up with 32 * 3 * 3 = 288 features. 

I have solved this problem by calculating the size of the activationmap with the ._conv_test method. After I calculate the size of the map (eg (2,2)) I can create an AvgPool2d layer that will take the average of the (2,2) map. This way you will always end up with (batch, filters, 1, 1) and after the flatten this will be filter * 1 * 1, which is exactly the amount of filters.

In [25]:
avgpool = nn.AvgPool2d((2,2))
pooled = avgpool(out)
pooled.shape

torch.Size([64, 32, 1, 1])

If we flatten this, we obtain 32x1x1 numbers, which is still 32, which makes designing your model a bit easier (and you might also argue that taking the average is a good approach in terms of model logic)

Let's combine it all together, and add a _conv_test method to create the right size for the AvgPool2D layer.

In [26]:
import torch
from torch import nn
from loguru import logger
from torchinfo import summary
import copy


# Define model
class CNN(nn.Module):
    def __init__(self, filters: int, units1: int, units2: int, input_size: tuple):
        super().__init__()
        self.in_channels = input_size[1] # (batch x channels x height x width), so we need the second element
        self.input_size = input_size
        self.filters = filters
        self.units1 = units1
        self.units2 = units2

        self.convolutions = nn.Sequential(
            nn.Conv2d(self.in_channels, filters, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(filters, filters, kernel_size=3, stride=1, padding=0),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(filters, filters, kernel_size=3, stride=1, padding=0),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )

        activation_map_size = self._conv_test(self.input_size)
        logger.info(f"Aggregating activationmap with size {activation_map_size}")
        self.agg = nn.AvgPool2d(activation_map_size)

        self.dense = nn.Sequential(
            nn.Flatten(),
            nn.Linear(filters, units1),
            nn.ReLU(),
            nn.Linear(units1, units2),
            nn.ReLU(),
            nn.Linear(units2, 10)
        )

    def _conv_test(self, input_size):
        x = torch.ones(input_size, dtype=torch.float32)
        x = self.convolutions(x)
        return x.shape[-2:]

    def forward(self, x):
        x = self.convolutions(x)
        x = self.agg(x)
        logits = self.dense(x)
        return logits


In [ ]:
"""   
torchinfo.summary()     literally summerizes the output shape per step in the sequancies (what I did in a previous codeblock)
"""

model = CNN(filters=128, units1=128, units2=64, input_size=(32, 3, 224, 224))
summary(model, input_size=(32, 3, 224, 224), device="cpu")

2026-05-31 18:14:19.614 | INFO     | __main__:__init__:31 - Aggregating activationmap with size torch.Size([26, 26])


Layer (type:depth-idx)                   Output Shape              Param #
CNN                                      [32, 10]                  --
├─Sequential: 1-1                        [32, 128, 26, 26]         --
│    └─Conv2d: 2-1                       [32, 128, 224, 224]       3,584
│    └─ReLU: 2-2                         [32, 128, 224, 224]       --
│    └─MaxPool2d: 2-3                    [32, 128, 112, 112]       --
│    └─Conv2d: 2-4                       [32, 128, 110, 110]       147,584
│    └─ReLU: 2-5                         [32, 128, 110, 110]       --
│    └─MaxPool2d: 2-6                    [32, 128, 55, 55]         --
│    └─Conv2d: 2-7                       [32, 128, 53, 53]         147,584
│    └─ReLU: 2-8                         [32, 128, 53, 53]         --
│    └─MaxPool2d: 2-9                    [32, 128, 26, 26]         --
├─AvgPool2d: 1-2                         [32, 128, 1, 1]           --
├─Sequential: 1-3                        [32, 10]                  --
│ 

In [28]:
model = CNN(filters=128, units1=128, units2=64, input_size=(32, 1, 28, 28))
summary(model, input_size=(32, 1, 28, 28), device="cpu")

2026-05-31 18:29:41.405 | INFO     | __main__:__init__:31 - Aggregating activationmap with size torch.Size([2, 2])


Layer (type:depth-idx)                   Output Shape              Param #
CNN                                      [32, 10]                  --
├─Sequential: 1-1                        [32, 128, 2, 2]           --
│    └─Conv2d: 2-1                       [32, 128, 28, 28]         1,280
│    └─ReLU: 2-2                         [32, 128, 28, 28]         --
│    └─MaxPool2d: 2-3                    [32, 128, 14, 14]         --
│    └─Conv2d: 2-4                       [32, 128, 12, 12]         147,584
│    └─ReLU: 2-5                         [32, 128, 12, 12]         --
│    └─MaxPool2d: 2-6                    [32, 128, 6, 6]           --
│    └─Conv2d: 2-7                       [32, 128, 4, 4]           147,584
│    └─ReLU: 2-8                         [32, 128, 4, 4]           --
│    └─MaxPool2d: 2-9                    [32, 128, 2, 2]           --
├─AvgPool2d: 1-2                         [32, 128, 1, 1]           --
├─Sequential: 1-3                        [32, 10]                  --
│ 

We have about 15k parameters. You will always need to judge that relative to your input data: 

- how many observations do you have? 
- maybe even more important: how many features do you have? Images sized 28x28 will need much less complexity than images sized 224x224 (note how the first one has 784 features, the second one more than 50.000!)
- Do you think the model needs a lot of complexity, or not so much? E.g. classifying if there is a stamp, or not, on a piece of paper is much easier than classifying the age of a face.

Also think about:
What is the trade off between adding more complexity? Or reducing complexity?

Try to answer this trade of in terms of:

- speed
- generalization
- accuracy

Eg 512 filters might add 0.1 % accuracy, but it might double training time. Is that worth it? Often, not...

We will need to tell the model how good it is performing. To do that, we will need to pick a loss function $\mathcal{L}$. We will discuss this in more depth, but for now, just take my word for it that a CrossEntropyLoss is a good pick.

In [29]:
import torch.optim as optim
from mltrainer import metrics, Trainer
optimizer = optim.Adam
loss_fn = torch.nn.CrossEntropyLoss()
accuracy = metrics.Accuracy()

In [30]:
model = CNN(filters=128, units1=128, units2=64, input_size=(32, 1, 28, 28))

2026-05-31 18:31:34.349 | INFO     | __main__:__init__:31 - Aggregating activationmap with size torch.Size([2, 2])


In [31]:
yhat = model(x)
accuracy(y, yhat)

0.125

In [32]:
log_dir = Path("modellog").resolve()
if not log_dir.exists():
    log_dir.mkdir(parents=True)

We now have everything we need to train the model.

In [33]:
from mltrainer import TrainerSettings, ReportTypes

settings = TrainerSettings(
    epochs=3,
    metrics=[accuracy],
    logdir=log_dir,
    train_steps=len(train),
    valid_steps=len(valid),
    reporttypes=[ReportTypes.TENSORBOARD, ReportTypes.TOML],
)
settings

epochs: 3
metrics: [Accuracy]
logdir: /home/gjongde/MADS-MachineLearning-course/notebooks/2_convolutions/modellog
train_steps: 937
valid_steps: 156
reporttypes: [<ReportTypes.TENSORBOARD: 'TENSORBOARD'>, <ReportTypes.TOML: 'TOML'>]
optimizer_kwargs: {'lr': 0.001, 'weight_decay': 1e-05}
scheduler_kwargs: {'factor': 0.1, 'patience': 10}
earlystop_kwargs: {'save': False, 'verbose': True, 'patience': 10}

In [34]:
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
    print("Using MPS")
elif torch.cuda.is_available():
    device = "cuda:0"
    print("using cuda")
else:
    device = "cpu"
    print("using cpu")

using cpu


In [35]:
trainer = Trainer(
    model=model,
    settings=settings,
    loss_fn=loss_fn,
    optimizer=optimizer,
    traindataloader=trainstreamer,
    validdataloader=validstreamer,
    scheduler=optim.lr_scheduler.ReduceLROnPlateau,
    device=device,
    )

2026-05-31 18:32:18.051 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to /home/gjongde/MADS-MachineLearning-course/notebooks/2_convolutions/modellog/20260531-183218
2026-05-31 18:32:20.386 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.


In [36]:
trainer.loop()

100%|██████████| 937/937 [02:15<00:00,  6.90it/s]
2026-05-31 18:34:48.022 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 0.7204 test 0.4888 metric ['0.8172']
100%|██████████| 937/937 [02:11<00:00,  7.10it/s]
2026-05-31 18:37:08.634 | INFO     | mltrainer.trainer:report:209 - Epoch 1 train 0.4249 test 0.3840 metric ['0.8596']
100%|██████████| 937/937 [02:10<00:00,  7.15it/s]
2026-05-31 18:39:26.892 | INFO     | mltrainer.trainer:report:209 - Epoch 2 train 0.3381 test 0.3277 metric ['0.8813']
100%|██████████| 3/3 [07:01<00:00, 140.65s/it]


If you have version 0.1.129 of `mltrainer`, have a look at the `imagemodels.py` file. There you can find this model, but also a model that uses a more modular strategy (see the `ConvBlock` and `CNNBlocks` architectures)